# Lecture: Variational Autoencoder (VAE)

The standard autoencoder (notebook C2-1) maps each input to a **single point** in latent space. This works well for reconstruction and compression, but makes the latent space unreliable for *generation*: there is no guarantee that an arbitrary point $\mathbf{z}$ sampled from some distribution actually decodes to a realistic image.

A **Variational Autoencoder** (Kingma & Welling, 2013) fixes this by making the encoder *probabilistic*. Instead of a point, the encoder produces the parameters of a Gaussian posterior:

$$q_\psi(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}_\psi(\mathbf{x}),\, \text{diag}(\boldsymbol{\sigma}^2_\psi(\mathbf{x})))$$

Training maximises the **Evidence Lower BOund (ELBO)**:

$$ELBO(\psi, \theta) = \underbrace{\mathbb{E}_{q_\psi(\mathbf{z}|\mathbf{x})}[\log p_\theta(\mathbf{x} \mid \mathbf{z}])}_{\text{reconstruction}} - \underbrace{D_{KL}\bigl(q_\psi(\mathbf{z}|\mathbf{x}),\, p_\theta(\mathbf{z})\bigr)}_{\text{regularisation}}$$

- **Reconstruction term**: the decoder should reproduce the input well — identical to the AE objective.
- **KL term**: the posterior $q_\psi(\mathbf{z}|\mathbf{x})$ is pulled towards the standard normal prior $p_\theta(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$. This regularises the latent space and enables sampling at test time.

Because sampling from $q_\psi$ is non-differentiable, we use the **reparametrisation trick**:

$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

This moves the randomness into a fixed noise variable $\boldsymbol{\varepsilon}$, so gradients flow through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$ as usual.

For the Gaussian case, the KL term has a closed-form solution per latent dimension $j$:

$$D_{KL} = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/C2-Autoencoders/VAE.py ./

### Data Preparation

We use the full MNIST training set (60,000 images), identical to the autoencoder notebook. Pixel values are normalised to $[0, 1]$.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

### Model Architecture

The VAE shares the same convolutional backbone as the autoencoder from notebook C2-1. The key differences are in the encoder output:

- **Autoencoder encoder**: one linear head → $\mathbf{z}$
- **VAE encoder**: two linear heads → $\boldsymbol{\mu}$ and $\log \boldsymbol{\sigma}^2$

The decoder is identical. The `forward()` method additionally returns $\boldsymbol{\mu}$ and $\log \boldsymbol{\sigma}^2$ so the loss function can compute the KL term.

In [ ]:
from VAE import VAE

LATENT_DIM = 2

_vae = VAE(latent_dim=LATENT_DIM)
_x   = torch.zeros(4, 1, 28, 28)
_x_hat, _mu, _log_var = _vae(_x)

print("Reconstruction shape:", _x_hat.shape)    # (4, 1, 28, 28)
print("mu shape:            ", _mu.shape)        # (4, 2)
print("log_var shape:       ", _log_var.shape)   # (4, 2)

### ELBO Loss

The total loss is the **negative ELBO**, i.e. we *minimise*:

$$-ELBO = \underbrace{-\mathbb{E}_{q_\psi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction loss}} + \underbrace{D_{KL}\bigl(q_\psi(\mathbf{z}|\mathbf{x}),\, p_\theta(\mathbf{z})\bigr)}_{\text{KL regularisation}}$$

For a Gaussian decoder $p_\theta(x|z)$ the reconstruction term reduces to the MSE. The closed-form KL for a diagonal Gaussian posterior $q_\psi = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$ against $p_\theta(z) = \mathcal{N}(0, I)$ is:

$$D_{KL}\bigl(q_\psi(z|x),\, p_\theta(z)\bigr) = -\frac{1}{2}\sum_j\left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

We scale the KL by $\frac{1}{N}$ (number of pixels $N = 784$) so it is in the same units as the per-pixel MSE.

In [ ]:
import torch.nn.functional as F

def elbo_loss(
    x: torch.Tensor,
    x_hat: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute the negative ELBO as a sum of reconstruction loss and KL divergence.

    The KL term is scaled by the number of input pixels so both terms are
    in the same units as the per-pixel MSE.

    Returns:
        total  — sum of recon_loss and kl_loss (scalar to backprop)
        recon  — MSE reconstruction loss (for logging)
        kl     — KL divergence term (for logging)
    """
    n_pixels = x.shape[1] * x.shape[2] * x.shape[3]  # 1*28*28 = 784

    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl    = (-0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(dim=1)).mean() / n_pixels

    return recon + kl, recon, kl

### Training

We log both the reconstruction and KL term separately each epoch. At the start of training the KL term is close to zero — the encoder outputs an approximately standard normal posterior because the weights are initialised near zero. As training progresses, the reconstruction term decreases while the KL term rises slightly, reflecting the trade-off the model learns to balance.

In [ ]:
import torch.optim as optim

model     = VAE(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs    = 20

for epoch in range(epochs):
    model.train()
    total_loss = total_recon = total_kl = 0

    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)

        optimizer.zero_grad()
        x_hat, mu, log_var = model(x)
        loss, recon, kl = elbo_loss(x, x_hat, mu, log_var)
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        total_recon += recon.item()
        total_kl    += kl.item()

    n = len(train_loader)
    print(f"Epoch {epoch+1:2d}  "
          f"loss={total_loss/n:.5f}  "
          f"recon={total_recon/n:.5f}  "
          f"kl={total_kl/n:.5f}")

In [ ]:
model.save_model()

If you do not want to train, you can load the pre-trained model (latent_dim=2, 20 epochs, full MNIST training set).

In [ ]:
import torch
from VAE import VAE

device     = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 2

model = VAE(latent_dim=LATENT_DIM).to(device)
#model.load_model(path="models/vae_mnist.pth", device=device) # for running locally
model.load_model(path="AIBIP/C2-Autoencoders/models/vae_mnist.pth", device=device) # for running in colab

### Reconstruction Quality

We compare originals and reconstructions for the first 10 test images. At `latent_dim=2` the VAE bottleneck is very tight; slight blurring is expected and is a direct consequence of the KL regularisation pushing posteriors towards the prior.

In [ ]:
import matplotlib.pyplot as plt

model.eval()

x_batch, _ = next(iter(test_loader))
x_batch    = x_batch[:10].to(device)

with torch.no_grad():
    x_hat, _, _ = model(x_batch)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i].squeeze().cpu(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",       fontsize=10)
axes[1, 0].set_ylabel("Reconstruction", fontsize=10)
plt.tight_layout()
plt.show()

### Latent Space Visualisation

We encode all 10,000 test images and plot $\boldsymbol{\mu}$ — the posterior mean — in 2D, coloured by digit class. Compared to the plain autoencoder:

- Clusters are **smoother and more overlapping** because the KL term prevents the encoder from collapsing all representations to isolated points.
- The occupied region is roughly centred at the origin and has unit-scale spread, consistent with the $\mathcal{N}(\mathbf{0}, \mathbf{I})$ prior.

In [ ]:
import numpy as np

model.eval()

all_mu     = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        mu, _ = model.encoder(x.to(device))
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

all_mu     = np.concatenate(all_mu,     axis=0)  # (10000, 2)
all_labels = np.concatenate(all_labels, axis=0)  # (10000,)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(all_mu[:, 0], all_mu[:, 1],
                     c=all_labels, cmap="tab10", s=2, alpha=0.6)
plt.colorbar(scatter, ax=ax, label="Digit class", ticks=range(10))
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("2D Latent Space (posterior means) — MNIST Test Set")
plt.tight_layout()
plt.show()

### Latent Space Grid Decoding

Because the prior is $\mathcal{N}(\mathbf{0}, \mathbf{I})$, the latent space has a known scale. We can sweep a uniform grid over $[-3, 3]^2$ and decode each point, revealing how the VAE has organised digit morphology across the 2D plane.

In [ ]:
model.eval()

n_side  = 15
z_range = torch.linspace(-3, 3, n_side)

# Build grid of (n_side^2, 2) latent points
grid_z = torch.stack(
    torch.meshgrid(z_range, z_range, indexing="ij"), dim=-1
).reshape(-1, 2).to(device)

with torch.no_grad():
    imgs = model.decode(grid_z).squeeze(1).cpu().numpy()  # (n_side^2, 28, 28)

canvas = np.zeros((n_side * 28, n_side * 28))
for idx, img in enumerate(imgs):
    row = idx // n_side
    col = idx  % n_side
    canvas[row*28:(row+1)*28, col*28:(col+1)*28] = img

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(canvas, cmap="gray", extent=[-3, 3, -3, 3], origin="upper")
ax.set_xlabel("z[0]")
ax.set_ylabel("z[1]")
ax.set_title("Decoded images on a 15×15 grid in latent space")
plt.tight_layout()
plt.show()

### Sampling from the Prior

This is the key capability the plain autoencoder *lacks*: we can draw $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decode to obtain new, previously unseen images. The KL regularisation ensures that points drawn from the prior land in regions the decoder has been trained on.

In [ ]:
model.eval()

n_samples = 16
samples   = model.sample(n_samples, device=device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("Samples from p(z) = N(0, I) decoded by the VAE", y=1.02)
plt.tight_layout()
plt.show()

### Latent Space Interpolation

As with the autoencoder, we can linearly interpolate between the posterior means of two test images. The VAE interpolation is typically smoother because the KL term encourages a continuous, gap-free latent space.

In [ ]:
model.eval()

test_images, test_labels = next(iter(test_loader))

idx_a = (test_labels == 1).nonzero(as_tuple=True)[0][0]
idx_b = (test_labels == 7).nonzero(as_tuple=True)[0][0]

x_a = test_images[idx_a].unsqueeze(0).to(device)
x_b = test_images[idx_b].unsqueeze(0).to(device)

with torch.no_grad():
    mu_a, _ = model.encoder(x_a)
    mu_b, _ = model.encoder(x_b)

n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = (1 - alpha) * mu_a + alpha * mu_b
        img = model.decode(z_interp).squeeze().cpu()
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}", fontsize=8)

plt.suptitle(f"Interpolation: digit {test_labels[idx_a].item()} → digit {test_labels[idx_b].item()}", y=1.05)
plt.tight_layout()
plt.show()

---
## Exercise: The Effect of the Latent Dimension

So far everything used `latent_dim = 2` — convenient because the latent space
can be drawn as a 2D scatter plot. But two numbers are very little to describe a
handwritten digit, which is why the reconstructions look smooth and slightly
washed out.

In this exercise you compare the trained `latent_dim = 2` model with a second
model trained identically but with **`latent_dim = 16`**. Both checkpoints are
provided — **no training required**, just load and observe.

Work through the cells below and answer these questions:

1. **Reconstruction quality:** Which model reconstructs the digits more
   faithfully? Why does more latent capacity help?
2. **The trade-off:** The 2D model lets us *draw* the latent space (the scatter
   plot above) and *decode a grid* over it. Can you make the same 2D scatter
   plot for the 16-dimensional model? Why not — and what would you have to do to
   visualise it anyway?
3. **Sampling:** Generate prior samples from both models. Does the higher latent
   dimension change the variety or quality of generated digits?
4. **When would you pick which?** Name one situation where `latent_dim = 2` is
   the better choice despite worse reconstructions.

In [ ]:
from VAE import VAE

# Load both pre-trained models. The 2D model is the one used above;
# vae_mnist_latent16.pth was trained identically but with latent_dim=16.
vae2  = VAE(latent_dim=2).to(device)
#vae2.load_model(path="models/vae_mnist.pth", device=device) # for running locally
vae2.load_model(path="AIBIP/C2-Autoencoders/models/vae_mnist.pth", device=device) # for running in colab
vae2.eval()

vae16 = VAE(latent_dim=16).to(device)
#vae16.load_model(path="models/vae_mnist_latent16.pth", device=device) # for running locally
vae16.load_model(path="AIBIP/C2-Autoencoders/models/vae_mnist_latent16.pth", device=device) # for running in colab
vae16.eval()

In [ ]:
# Q1: Compare reconstructions of the same digits side by side.
x_batch, _ = next(iter(test_loader))
x_batch    = x_batch[:10].to(device)

with torch.no_grad():
    x2, _, _  = vae2(x_batch)
    x16, _, _ = vae16(x_batch)

fig, axes = plt.subplots(3, 10, figsize=(15, 4.5))
for i in range(10):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(x2[i].squeeze().cpu(),      cmap="gray"); axes[1, i].axis("off")
    axes[2, i].imshow(x16[i].squeeze().cpu(),     cmap="gray"); axes[2, i].axis("off")
axes[0, 0].set_ylabel("Original",        fontsize=10)
axes[1, 0].set_ylabel("latent_dim = 2",  fontsize=10)
axes[2, 0].set_ylabel("latent_dim = 16", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Q2: The 2D model's latent space is directly plottable (z[0] vs z[1]).
# For latent_dim=16 there is no canonical 2D view — we must *project* the 16D
# posterior means down to 2D. We use PCA, implemented here with a plain SVD
# (no extra dependency): the first two principal directions capture the
# directions of largest variance.
import numpy as np

mu16, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        mu, _ = vae16.encoder(x.to(device))
        mu16.append(mu.cpu()); labels.append(y)
mu16   = torch.cat(mu16, dim=0)                 # (10000, 16)
labels = torch.cat(labels, dim=0).numpy()

# PCA via SVD: centre the data, then project onto the top-2 right singular vectors.
mu_centered = mu16 - mu16.mean(dim=0, keepdim=True)
_, _, V = torch.linalg.svd(mu_centered, full_matrices=False)
mu16_2d = (mu_centered @ V[:2].T).numpy()       # (10000, 2)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(mu16_2d[:, 0], mu16_2d[:, 1], c=labels, cmap="tab10", s=2, alpha=0.6)
plt.colorbar(sc, ax=ax, label="Digit class", ticks=range(10))
ax.set_xlabel("PCA component 1"); ax.set_ylabel("PCA component 2")
ax.set_title("16D Latent Space projected to 2D via PCA (SVD) — MNIST Test Set")
plt.tight_layout()
plt.show()

In [ ]:
# Q3: Sample from the prior of both models and compare generated digits.
n = 10
with torch.no_grad():
    s2  = vae2.sample(n,  device=device)
    s16 = vae16.sample(n, device=device)

fig, axes = plt.subplots(2, n, figsize=(15, 3))
for i in range(n):
    axes[0, i].imshow(s2[i].squeeze().cpu(),  cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(s16[i].squeeze().cpu(), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("dim = 2",  fontsize=10)
axes[1, 0].set_ylabel("dim = 16", fontsize=10)
plt.suptitle("Prior samples  z ~ N(0, I)")
plt.tight_layout()
plt.show()